## Motivation and modeling rationale

Preliminary analysis of the development set reveals that the dataset exhibits a strong structural heterogeneity. Alongside genuinely ambiguous documents that require semantic interpretation, a substantial portion of the data contains highly repetitive lexical and editorial patterns that are almost perfectly predictive of the target class.

A baseline linear model trained with a standard architecture (source identity, word- and character-level TF-IDF, and simple numeric features) already highlights this behavior through feature importance analysis: a small number of features dominate the decision function, while the majority contribute marginally. This indicates the presence of low-entropy signals that are implicitly learned by the probabilistic model but treated in the same way as weaker, noisy features.

To make this behavior explicit, we analyze lexical tokens and editorial markers separately. This analysis shows that several words, n-grams, and HTML or layout-related tags exhibit near-perfect determinism, with purity equal to 1 and zero entropy across hundreds or thousands of documents. In practice, the presence of a single such pattern is sufficient to determine the class with extremely high confidence, independently of the surrounding context.

These findings suggest that the dataset naturally decomposes into two distinct regimes: (i) documents that can be classified deterministically based on the presence of highly specific patterns, and (ii) documents that lack such cues and require probabilistic reasoning over weaker semantic and stylistic signals. A single-stage classifier necessarily averages these two regimes, leading to suboptimal modeling of both.

This observation directly motivates the adoption of a two-stage strategy. The first stage explicitly handles deterministic cases through high-purity lexical and editorial rules, while the second stage focuses exclusively on the remaining ambiguous instances using a statistical classifier. The two-stage model therefore reflects the intrinsic structure of the dataset rather than introducing additional assumptions or ad-hoc complexity.


In [2]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score


In [ ]:
# 
# BASELINE PARAMETERS (NON TUNED)
# 

WORD_NG_MAX = 2
CHAR_NG_MAX = 5
MIN_DF      = 2
MAX_DF      = 0.9
C_VALUE     = 1.5

NUM_COLS = [
	"n_tokens",
	"title_len",
	"article_len",
	"title_ratio"
]


In [4]:
DEV_PATH = "../data/raw/development.csv"
df = pd.read_csv(DEV_PATH)

for col in ["title", "article", "source"]:
	df[col] = df[col].fillna("").astype(str)

df["text"] = (df["title"] + " " + df["article"]).str.lower()

label_col = "label"


In [5]:
df["n_tokens"]    = df["text"].str.split().str.len()
df["title_len"]   = df["title"].str.len()
df["article_len"] = df["article"].str.len()
df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)


In [6]:
X = df[["source", "text"] + NUM_COLS]
y = df[label_col]

X_tr, X_te, y_tr, y_te = train_test_split(
	X,
	y,
	test_size=0.2,
	random_state=42,
	stratify=y
)


In [7]:
baseline_model = Pipeline([
	("pre", ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),

			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, WORD_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=250_000
			), "text"),

			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, CHAR_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),

			("num", StandardScaler(), NUM_COLS),
		],
		remainder="drop",
		n_jobs=-1
	)),
	("clf", LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	))
])


In [8]:
baseline_model.fit(X_tr, y_tr)
pred = baseline_model.predict(X_te)

print("Baseline Macro F1:",
	  f1_score(y_te, pred, average="macro"))


Baseline Macro F1: 0.716546127237697


In [9]:
pre = baseline_model.named_steps["pre"]
clf = baseline_model.named_steps["clf"]

feature_names = []

# source OHE
src_names = pre.named_transformers_["src"].get_feature_names_out(["source"])
feature_names.extend(src_names)

# word tfidf
word_names = pre.named_transformers_["w_tfidf"].get_feature_names_out()
feature_names.extend(word_names)

# char tfidf
char_names = pre.named_transformers_["c_tfidf"].get_feature_names_out()
feature_names.extend(char_names)

# numeric
feature_names.extend(NUM_COLS)

feature_names = np.array(feature_names)


In [10]:
coefs = clf.coef_  # shape: (n_classes, n_features)

# importance = mean absolute weight across classes
importance = np.mean(np.abs(coefs), axis=0)

imp_df = pd.DataFrame({
	"feature": feature_names,
	"importance": importance
}).sort_values("importance", ascending=False)


In [11]:
imp_df.head(30)


,feature,importance
843,source_RedNova,2.949948
1009,source_Topix.Net,1.824104
1166,source_Wired,1.688497
256820,\,1.634857
163,source_CSMonitor,1.586897
96647,health,1.501349
1000,source_Time,1.425754
128,source_Boston,1.375833
78561,entertainment http,1.280731
194173,rss entertainment,1.280731


In [12]:
for i, cls in enumerate(clf.classes_):
	print(f"\n=== TOP FEATURES | CLASS {cls} ===")
	tmp = pd.DataFrame({
		"feature": feature_names,
		"weight": coefs[i]
	})
	tmp["abs_weight"] = tmp["weight"].abs()
	display(tmp.sort_values("abs_weight", ascending=False).head(15))



=== TOP FEATURES | CLASS 0 ===


,feature,weight,abs_weight
1009,source_Topix.Net,6.384363,6.384363
163,source_CSMonitor,4.275078,4.275078
284274,..,3.778096,3.778096
284280,...,3.758757,3.758757
9766,afp,3.693784,3.693784
459,source_IPS,3.540019,3.540019
284279,...,3.293367,3.293367
843,source_RedNova,3.094925,3.094925
10072,afp afp,2.686838,2.686838
194189,rss world,2.676434,2.676434



=== TOP FEATURES | CLASS 1 ===


,feature,weight,abs_weight
194168,rss business,3.921688,3.921688
681,source_Motley,3.859810,3.859810
1166,source_Wired,3.232154,3.232154
55,source_Ananova,3.125123,3.125123
42899,business,3.054415,3.054415
42972,business http,2.898538,2.898538
18893,ap,-2.877208,2.877208
478,source_InfoWorld,2.547042,2.547042
843,source_RedNova,-2.388445,2.388445
156990,oil,2.279451,2.279451



=== TOP FEATURES | CLASS 2 ===


...

,feature,weight,abs_weight
212115,sports,3.911853,3.911853
194183,rss sports,3.552458,3.552458
843,source_RedNova,-3.116499,3.116499
18893,ap,2.970745,2.970745
212137,sports http,2.858389,2.858389
221611,team,2.596259,2.596259
96228,he,2.334595,2.334595
64879,cup,2.276400,2.276400
55564,coach,2.247871,2.247871
852,source_Renault,2.202927,2.202927



=== TOP FEATURES | CLASS 5 ===


,feature,weight,abs_weight
1179,source_Yahoo,-3.444552,3.444552
1191,source_\,3.340312,3.340312
250538,york reuters,-3.006228,3.006228
231303,time topstories,2.606396,2.606396
234997,topstories,2.606396,2.606396
843,source_RedNova,-2.556991,2.556991
29521,baghdad reuters,2.218302,2.218302
413,source_Guardian,2.151182,2.151182
55,source_Ananova,-2.032365,2.032365
9380,adv,1.872941,1.872941



=== TOP FEATURES | CLASS 6 ===


,feature,weight,abs_weight
96647,health,5.086523,5.086523
194176,rss health,3.959573,3.959573
47006,cancer,3.510182,3.510182
11659,aids,3.143989,3.143989
96685,health http,3.100313,3.100313
415311,heal,2.827824,2.827824
415317,healt,2.627482,2.627482
388037,ealt,2.455399,2.455399
388039,ealth,2.449312,2.449312
353977,alth,2.365204,2.365204


In [13]:
import re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd


In [14]:
def tokenize_words(text):
	return re.findall(r"\b\w+\b", text.lower())

def tokenize_ngrams(tokens, n):
	return zip(*[tokens[i:] for i in range(n)])


In [15]:
def lexical_determinism(df, ngram_max=2, min_support=30):
	label_counts = Counter(df["label"])
	n_docs = len(df)

	token_label_counts = defaultdict(lambda: Counter())
	token_doc_count = Counter()

	for _, row in df.iterrows():
		tokens = tokenize_words(row["text"])
		seen = set()

		for n in range(1, ngram_max + 1):
			for ng in tokenize_ngrams(tokens, n):
				token = "_".join(ng)
				if token not in seen:
					token_label_counts[token][row["label"]] += 1
					token_doc_count[token] += 1
					seen.add(token)

	rows = []

	for token, lbl_cnt in token_label_counts.items():
		support = token_doc_count[token]
		if support < min_support:
			continue

		total = sum(lbl_cnt.values())
		probs = np.array(list(lbl_cnt.values())) / total

		entropy = -(probs * np.log2(probs)).sum()
		purity = probs.max()
		dominant_label = max(lbl_cnt, key=lbl_cnt.get)

		rows.append({
			"token": token,
			"type": "lexical",
			"support": support,
			"purity": purity,
			"entropy": entropy,
			"dominant_label": dominant_label
		})

	return pd.DataFrame(rows).sort_values(
		["purity", "support"],
		ascending=[False, False]
	)


In [21]:
lex_df = lexical_determinism(df, ngram_max=WORD_NG_MAX, min_support=30)
lex_df.head(20)


,token,type,support,purity,entropy,dominant_label
2890,rss_world,lexical,1494,1.0,-0.0,0
2891,world_http,lexical,1438,1.0,-0.0,0
235,rss_europe,lexical,1115,1.0,-0.0,0
236,europe_http,lexical,1115,1.0,-0.0,0
1524,rss_entertainment,lexical,1022,1.0,-0.0,3
1525,entertainment_http,lexical,1022,1.0,-0.0,3
1469,rss_sports,lexical,1010,1.0,-0.0,4
1470,sports_http,lexical,896,1.0,-0.0,4
1413,0_alt,lexical,644,1.0,-0.0,2
1574,h4,lexical,628,1.0,-0.0,2


In [ ]:
def extract_html_tags(text):
	return re.findall(r"<\/?\w+[^>]*>", text.lower())

def extract_editorial_patterns(text):
	patterns = []
	if "<img" in text: patterns.append("HAS_IMG")
	if "<a " in text: patterns.append("HAS_LINK")
	if "<iframe" in text: patterns.append("HAS_IFRAME")
	if "class=" in text: patterns.append("HAS_CLASS_ATTR")
	if "style=" in text: patterns.append("HAS_STYLE_ATTR")
	return patterns


In [20]:
def tag_determinism(df, min_support=30):
	tag_label_counts = defaultdict(lambda: Counter())
	tag_doc_count = Counter()

	for _, row in df.iterrows():
		tags = set(extract_html_tags(row["article"]))
		patterns = set(extract_editorial_patterns(row["article"]))
		all_tags = tags.union(patterns)

		for tag in all_tags:
			tag_label_counts[tag][row["label"]] += 1
			tag_doc_count[tag] += 1

	rows = []

	for tag, lbl_cnt in tag_label_counts.items():
		support = tag_doc_count[tag]
		if support < min_support:
			continue

		total = sum(lbl_cnt.values())
		probs = np.array(list(lbl_cnt.values())) / total

		entropy = -(probs * np.log2(probs)).sum()
		purity = probs.max()
		dominant_label = max(lbl_cnt, key=lbl_cnt.get)

		rows.append({
			"token": tag,
			"type": "editorial",
			"support": support,
			"purity": purity,
			"entropy": entropy,
			"dominant_label": dominant_label
		})

	return pd.DataFrame(rows).sort_values(
		["purity", "support"],
		ascending=[False, False]
	)


In [19]:
tag_df = tag_determinism(df, min_support=30)
tag_df.head(20)


,token,type,support,purity,entropy,dominant_label
9,<h4>,editorial,628,1.000000,-0.000000,2
10,</h4>,editorial,628,1.000000,-0.000000,2
19,"<p align=""right"">",editorial,311,1.000000,-0.000000,2
18,"<a href=""http://www.infoworld.com/?source=rss"">",editorial,233,1.000000,-0.000000,2
6,"<img class=""0"" src=""http://adlog.com.com/adlog...",editorial,196,1.000000,-0.000000,2
7,"<img class=""rss-ad"" src=""http://adlog.com.com/...",editorial,196,1.000000,-0.000000,2
14,<a href='http://www.newsisfree.com/sources/inf...,editorial,156,1.000000,-0.000000,5
37,<strong>,editorial,111,1.000000,-0.000000,2
38,</strong>,editorial,111,1.000000,-0.000000,2
32,"<a href=""http://ad.doubleclick.net/jump/idg.us...",editorial,81,1.000000,-0.000000,2
